In [4]:
# ================================
# IMPORTS E CONEXÃO COM AZURE ML
# ================================

import mlflow
import mlflow.sklearn
from azureml.core import Workspace

# Conectar ao Workspace do Azure ML usando config.json
ws = Workspace.from_config(path="../config.json") 

print("Conectado ao Workspace Azure ML:", ws.name)

# Definir experimento MLflow no Azure
mlflow.set_experiment("SmartStock-Demand-Forecasting")
print("Experimento MLflow definido: SmartStock-Demand-Forecasting")

# ================================
# SMARTSTOCK AI - TRAINING PIPELINE
# ================================

# Importar bibliotecas principais

import pandas as pd
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Visualização
import matplotlib.pyplot as plt

print("Bibliotecas carregadas com sucesso.")

Conectado ao Workspace Azure ML: aml-smartstock-dp100


2026/02/17 19:58:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/17 19:58:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/17 19:58:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/17 19:58:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/17 19:58:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/17 19:58:25 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/17 19:58:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/17 19:58:26 INFO mlflow.store.db.utils: Updating database tables
2026/02/17 19:58:26 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/17 19:58:26 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/17 19:58:27 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/02/17 19:5

Experimento MLflow definido: SmartStock-Demand-Forecasting
Bibliotecas carregadas com sucesso.


In [5]:
# ================================
# CARREGAR DATASET
# ================================

data_path = "../data/sales_dataset.csv"
df = pd.read_csv(data_path)

df.head()
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   data           100 non-null    object 
 1   dia_semana     100 non-null    int64  
 2   fim_de_semana  100 non-null    int64  
 3   feriado        100 non-null    int64  
 4   promocao       100 non-null    int64  
 5   preco          100 non-null    float64
 6   temperatura    100 non-null    int64  
 7   clientes_loja  100 non-null    int64  
 8   marketing      100 non-null    int64  
 9   estoque        100 non-null    int64  
 10  vendas         100 non-null    int64  
dtypes: float64(1), int64(9), object(1)
memory usage: 8.7+ KB


data             0
dia_semana       0
fim_de_semana    0
feriado          0
promocao         0
preco            0
temperatura      0
clientes_loja    0
marketing        0
estoque          0
vendas           0
dtype: int64

In [6]:
# ================================
# INFORMAÇÕES DO DATASET
# ================================

print("Shape do dataset:", df.shape)

print("\nInformações gerais:")
df.info()

print("\nEstatísticas:")
df.describe()

Shape do dataset: (100, 11)

Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   data           100 non-null    object 
 1   dia_semana     100 non-null    int64  
 2   fim_de_semana  100 non-null    int64  
 3   feriado        100 non-null    int64  
 4   promocao       100 non-null    int64  
 5   preco          100 non-null    float64
 6   temperatura    100 non-null    int64  
 7   clientes_loja  100 non-null    int64  
 8   marketing      100 non-null    int64  
 9   estoque        100 non-null    int64  
 10  vendas         100 non-null    int64  
dtypes: float64(1), int64(9), object(1)
memory usage: 8.7+ KB

Estatísticas:


,dia_semana,fim_de_semana,feriado,promocao,preco,temperatura,clientes_loja,marketing,estoque,vendas
count,100.000000,100.000000,100.00,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,3.010000,0.280000,0.01,0.440000,28.500000,35.030000,284.550000,163.400000,102.400000,84.420000
std,1.992384,0.451261,0.10,0.498888,1.734964,4.212667,107.855812,127.598011,42.890982,43.172939
min,0.000000,0.000000,0.00,0.000000,25.900000,28.000000,150.000000,40.000000,55.000000,35.000000
25%,1.000000,0.000000,0.00,0.000000,25.900000,31.000000,200.000000,60.000000,73.750000,51.500000
50%,3.000000,0.000000,0.00,0.000000,29.900000,35.000000,250.000000,70.000000,80.000000,66.500000
75%,5.000000,1.000000,0.00,1.000000,29.900000,38.000000,352.500000,280.000000,130.000000,110.000000
max,6.000000,1.000000,1.00,1.000000,29.900000,45.000000,520.000000,420.000000,205.000000,185.000000


In [7]:
# ================================
# SEPARAR FEATURES E TARGET
# ================================

# Variável alvo
y = df["vendas"]

# Variáveis de entrada
X = df.drop(columns=["vendas", "data"])

print("Features:", X.columns.tolist())
print("Target: vendas")

Features: ['dia_semana', 'fim_de_semana', 'feriado', 'promocao', 'preco', 'temperatura', 'clientes_loja', 'marketing', 'estoque']
Target: vendas


In [8]:
# ================================
# DIVIDIR TREINO E TESTE
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (80, 9)
Teste: (20, 9)


In [9]:
# ================================
# TREINAR MODELO COM MLflow TRACKING
# ================================

from sklearn.linear_model import LinearRegression

with mlflow.start_run():  # inicia um run MLflow

    # Criar modelo
    model = LinearRegression()
    
    # Treinar
    model.fit(X_train, y_train)
    
    # Previsões
    predictions = model.predict(X_test)
    
    # Métricas
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    # Logar métricas no MLflow
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("R2", r2)
    
    # Logar modelo no MLflow + Azure ML
    mlflow.sklearn.log_model(sk_model=model, artifact_path="model", registered_model_name="smartstock_model")
    
    print("Modelo treinado e registrado no MLflow/Azure ML")

2026/02/17 19:58:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\Lucas_Drac0\Desktop\personal_projects\smartstock-ai\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Modelo treinado e registrado no MLflow/Azure ML


Successfully registered model 'smartstock_model'.
Created version '1' of model 'smartstock_model'.


In [10]:
# ================================
# FAZER PREVISÕES E AVALIAR PERFORMANCE
# ================================
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"MAE (Erro Médio Absoluto): {mae:.2f}")
print(f"R² Score: {r2:.2f}")

MAE (Erro Médio Absoluto): 2.55
R² Score: 0.99


In [11]:
# ================================
# TESTE COM NOVO DADO
# ================================
novo_dado = pd.DataFrame({
    "dia_semana": [6],
    "fim_de_semana": [1],
    "feriado": [0],
    "promocao": [1],
    "preco": [25.9],
    "temperatura": [35],
    "clientes_loja": [400],
    "marketing": [300],
    "estoque": [150]
})

pred = model.predict(novo_dado)
print(f"Previsão de vendas: {pred[0]:.0f} unidades")

Previsão de vendas: 131 unidades


In [12]:
print("Pipeline final executado com sucesso. Modelo registrado no Azure ML.")

Pipeline final executado com sucesso. Modelo registrado no Azure ML.
